# DNA Methylation Analysis with `cytozip`

CLI walkthrough using 9 mouse 3C-seq cells shipped under
`cytozip_example_data/bam/`. All steps are demonstrated as shell
commands (`! czip ...`); benchmark results produced offline by the
scripts under `tests/` are loaded and rendered as tables.

1. Setup + reference `.cz`
2. BAM → `.cz`  (`czip bam_to_cz`)
3. ALLC → `.cz` (`czip allc2cz`) + benchmark table (BAM→ALLC vs BAM→CZ, ALLC→CZ)
4. `view` / `query` (local) + benchmark table (tabix vs czip query)
5. `view` / `query` a remote `.cz`
6. `catcz` + `merge_cz`
7. `cz_to_anndata`

Benchmark
```python
    python tests/benchmark_bam_to_cz.py  -j 20
    python tests/benchmark_allc_to_cz.py -j 20
    python tests/benchmark_query.py
```

In [1]:
import os,sys
import pandas as pd
os.chdir(os.path.expanduser("~/Projects/Github/cytozip/cytozip_example_data"))

## 1. Build the reference `.cz`

The reference holds the genome-wide `(chrom, pos, strand, context)`
axis. Per-cell `.cz` then store only `mc`/`cov` and reuse the
reference's positions, cutting per-cell size by ~5×.


In [2]:
!czip build_ref --help

usage: czip build_ref [-h] -g GENOME [-O OUTPUT] [-p PATTERN] [-t THREADS]
                      [--keep_temp] [--no_delta]

options:
  -h, --help            show this help message and exit
  -g GENOME, --genome GENOME
                        reference genome FASTA (default: None)
  -O OUTPUT, --output OUTPUT
                        output .cz file (default: hg38_allc.cz)
  -p PATTERN, --pattern PATTERN
                        nucleotide pattern (default: C)
  -t THREADS, --threads THREADS
                        parallel jobs (default: 12)
  --keep_temp           keep temp directory (default: False)
  --no_delta            disable DELTA encoding on the pos column (default: on,
                        gives ~3x smaller reference files with mild query
                        overhead) (default: False)


In [3]:
!time czip build_ref -g ~/Ref/mm10/mm10_ucsc_with_chrL.fa -O output/mm10_with_chrL.allc.cz -t 20


2026-04-24 23:16:51.321 | DEBUG    | cytozip.allc:WriteC:62 - chr1
2026-04-24 23:16:52.000 | DEBUG    | cytozip.allc:WriteC:62 - chr10
2026-04-24 23:16:52.669 | DEBUG    | cytozip.allc:WriteC:62 - chr11
2026-04-24 23:16:53.338 | DEBUG    | cytozip.allc:WriteC:62 - chr12
2026-04-24 23:16:53.978 | DEBUG    | cytozip.allc:WriteC:62 - chr13
2026-04-24 23:16:54.645 | DEBUG    | cytozip.allc:WriteC:62 - chr14
2026-04-24 23:16:55.193 | DEBUG    | cytozip.allc:WriteC:62 - chr15
2026-04-24 23:16:55.707 | DEBUG    | cytozip.allc:WriteC:62 - chr16
2026-04-24 23:16:56.214 | DEBUG    | cytozip.allc:WriteC:62 - chr17
2026-04-24 23:16:56.703 | DEBUG    | cytozip.allc:WriteC:62 - chr18
2026-04-24 23:16:57.008 | DEBUG    | cytozip.allc:WriteC:62 - chr1_GL456210_random
2026-04-24 23:16:57.008 | DEBUG    | cytozip.allc:WriteC:62 - chr1_GL456211_random
2026-04-24 23:16:57.009 | DEBUG    | cytozip.allc:WriteC:62 - chr1_GL456212_random
2026-04-24 23:16:57.009 | DEBUG    | cytozip.allc:WriteC:62 - chr1_GL456

In [4]:
!czip header -I output/mm10_with_chrL.allc.cz

magic  :  b'CZIP'
version  :  0.3
total_size  :  1362568959
message  :  /home/x-wding2/Ref/mm10/mm10_ucsc_with_chrL.fa
formats  :  ['Q', 'c', '3s']
columns  :  ['pos', 'strand', 'context']
sort_col  :  0
delta_cols  :  [0]
chunk_dims  :  ['chrom']
header_size  :  102


In [5]:
! czip view -I output/mm10_with_chrL.allc.cz --show_dims 0 | head

chrom	pos	strand	context
chr1	3000003	+	CTG
chr1	3000005	-	CAG
chr1	3000009	+	CTA
chr1	3000016	-	CAA
chr1	3000018	-	CAC
chr1	3000019	-	CCA
chr1	3000023	+	CTT
chr1	3000027	-	CAA
chr1	3000029	-	CTC


## 3. BAM → `.cz`

Convert a position-sorted BAM directly to `.cz` (no intermediate ALLC
text). The output has only `mc` / `cov` because we pass a reference.


In [6]:
!czip bam_to_cz --help

usage: czip bam_to_cz [-h] -I INPUT -g GENOME [-O OUTPUT]
                      [--num_upstr_bases NUM_UPSTR_BASES]
                      [--num_downstr_bases NUM_DOWNSTR_BASES]
                      [--min_mapq MIN_MAPQ]
                      [--min_base_quality MIN_BASE_QUALITY] [-c BATCH_SIZE]
                      [--convert_bam_strandness] [--save_count_df]
                      [--mode {full,pos_mc_cov,mc_cov}]
                      [--count_fmt {B,H,I,Q}] [-r REFERENCE]

options:
  -h, --help            show this help message and exit
  -I INPUT, --input INPUT
                        input position-sorted BAM (bismark/hisat-3n) (default:
                        None)
  -g GENOME, --genome GENOME
                        indexed reference fasta (.fai required) (default:
                        None)
  -O OUTPUT, --output OUTPUT
                        output .cz path (default: <bam_stem>.cz) (default:
                        None)
  --num_upstr_bases NUM_UPSTR_BASES
              

In [7]:
! time czip bam_to_cz -I bam/FC_M_P12b_3C_2-5-M17-N10.hisat3n_dna.all_reads.deduped.bam \
 --genome ~/Ref/mm10/mm10_ucsc_with_chrL.fa -O output/cz/FC_M_P12b_3C_2-5-M17-N10.cz \
 --mode mc_cov --count_fmt B --reference output/mm10_with_chrL.allc.cz


/anvil/projects/x-mcb130189/Wubin/Github/cytozip/cytozip/__init__.py:558: UserWarning: mc/cov value exceeds count_fmt='B' max (255); clipping. Consider count_fmt='H' for bulk/high-coverage data.
  bam_to_cz(bam_path=args.input, genome=args.genome,

real	3m11.809s
user	3m46.321s
sys	0m11.266s


bam to allc
```python
from ALLCools._bam_to_allc import bam_to_allc
%time bam_to_allc(bam_path="bam/FC_M_P12b_3C_2-5-M17-N10.hisat3n_dna.all_reads.deduped.bam", reference_fasta=os.path.expanduser("~/Ref/mm10/mm10_ucsc_with_chrL.fa"), output_path="output/allc/FC_M_P12b_3C_2-5-M17-N10.allc.tsv.gz", cpu=1,num_upstr_bases=0, num_downstr_bases=2, min_mapq=10, min_base_quality=20, compress_level=5)
```

### BAM → `.cz` benchmark vs ALLCools

Produced by `tests/benchmark_bam_to_cz.py -j 9`. ALLCools writes a
gzipped TSV (`.allc.tsv.gz`); cytozip writes a chunked, columnar
zstd-compressed `.cz`.


In [8]:
import pandas as pd
bam_bench = pd.read_csv('output/bam_benchmark/bam_benchmark.tsv', sep='\t')
bam_bench.round(2)


,cell,bam_size_mb,n_reads,allc_wall_s,allc_rss_mb,allc_size_mb,cz_wall_s,cz_rss_mb,cz_size_mb,speedup,size_ratio
0,FC_E17b_3C_5-5-I24-A21,685.99,7330382,553.11,572.02,442.62,516.98,917.55,80.15,1.07,0.18
1,FC_M_E15a_3C_1-1-I5-B1,177.37,1787812,150.56,573.87,102.53,152.47,917.32,23.49,0.99,0.23
2,FC_M_P12b_3C_2-5-M17-N10,239.89,2307326,203.78,573.89,143.12,198.39,917.57,30.93,1.03,0.22
3,FC_M_P3b_3C_6-6-J3-P24,147.36,1395949,135.65,574.07,85.71,138.78,917.60,20.62,0.98,0.24
4,FC_M_P6a_3C_7-3-K21-P5,89.76,938870,93.69,574.47,48.84,95.97,917.56,13.39,0.98,0.27
5,FC_M_P9B_3C_6-2-F6-O4,32.28,342778,48.84,573.36,17.61,63.37,919.60,7.07,0.77,0.40
6,FC_P0b_3C_5-1-I24-J14,485.51,5028030,387.34,571.88,285.50,354.16,917.48,53.90,1.09,0.19
7,FC_P13a_3C_7-1-A11-O1,260.71,2588754,223.55,572.09,155.84,217.16,919.67,32.55,1.03,0.21
8,FC_P28a_3C_2-1-E5-N14,186.32,1908846,173.22,574.45,112.95,163.08,917.87,25.20,1.06,0.22


In [9]:
!cat output/bam_benchmark/bam_benchmark.txt

cytozip bam_to_cz  vs  ALLCools bam_to_allc
reference FASTA : /home/x-wding2/Ref/mm10/mm10_ucsc_with_chrL.fa
reference .cz   : /anvil/projects/x-mcb130189/Wubin/Github/cytozip/cytozip_example_data/output/mm10_with_chrL.allc.cz
BAMs            : 9   total reads = 23,628,747

ALLCools : time=  1969.7 s   size=  1394.7 MB
cytozip  : time=  1900.4 s   size=   287.3 MB

speedup (allc / cz time)  =  1.04x
compression (cz / allc)   =  20.6%


## 4. ALLC → `.cz`

Pack an existing `.allc.tsv.gz` into `.cz`. Two layouts:

In [10]:
! czip allc2cz --help

usage: czip allc2cz [-h] -I INPUT -O OUTPUT [-r REFERENCE]
                    [--missing_value MISSING_VALUE] [-F FORMATS] [-C COLUMNS]
                    [-D CHUNK_DIMS] [-u USECOLS] [--ref_pos_col REF_POS_COL]
                    [--allc_pos_col ALLC_POS_COL] [-s SEP]
                    [--chrom_order CHROM_ORDER] [-c BATCH_SIZE]
                    [--sort_col SORT_COL] [--delta_cols DELTA_COLS]

options:
  -h, --help            show this help message and exit
  -I INPUT, --input INPUT
                        input allc.tsv.gz (default: None)
  -O OUTPUT, --output OUTPUT
                        output .cz file (default: None)
  -r REFERENCE, --reference REFERENCE
                        reference .cz file (default: None)
  --missing_value MISSING_VALUE
                        missing value fill (default: [0, 0])
  -F FORMATS, --formats FORMATS
                        column formats (default: ['B', 'B'])
  -C COLUMNS, --columns COLUMNS
                        column names (default

In [12]:
# Standalone (positions inside the .cz)
! time czip allc2cz -I output/allc/FC_M_P12b_3C_2-5-M17-N10.allc.tsv.gz \
      -O output/allc/FC_M_P12b_3C_2-5-M17-N10.with_pos.cz \
      -F Q,B,B -C pos,mc,cov -u 1,4,5


2026-04-24 23:22:45.816 | INFO     | cytozip.allc:allc2cz:236 - /anvil/projects/x-mcb130189/Wubin/Github/cytozip/cytozip_example_data/output/allc/FC_M_P12b_3C_2-5-M17-N10.allc.tsv.gz

real	0m28.029s
user	0m24.823s
sys	0m2.393s


In [13]:
# Reference-aligned (positions come from the reference .cz)
! time czip allc2cz -I output/allc/FC_M_P12b_3C_2-5-M17-N10.allc.tsv.gz \
      -O output/allc/FC_M_P12b_3C_2-5-M17-N10.cz \
      -r output/mm10_with_chrL.allc.cz

2026-04-24 23:23:13.970 | INFO     | cytozip.allc:allc2cz:236 - /anvil/projects/x-mcb130189/Wubin/Github/cytozip/cytozip_example_data/output/allc/FC_M_P12b_3C_2-5-M17-N10.allc.tsv.gz

real	1m8.830s
user	0m58.523s
sys	0m10.016s


In [14]:
! ls output/allc/FC_M_P12b_3C_2-5-M17-N10* -sh

137M output/allc/FC_M_P12b_3C_2-5-M17-N10.allc.tsv.gz
1.5M output/allc/FC_M_P12b_3C_2-5-M17-N10.allc.tsv.gz.tbi
 30M output/allc/FC_M_P12b_3C_2-5-M17-N10.cz
 43M output/allc/FC_M_P12b_3C_2-5-M17-N10.with_pos.cz


## 5. `view` and `query`

`czip view` streams an entire dimension (e.g. one chrom). `czip query`
pulls a region. Both can use `-r REFERENCE` to expand reference-aligned
files into full ALLC-style records (chrom / pos / context / mc / cov).


In [15]:
! czip view -I output/allc/FC_M_P12b_3C_2-5-M17-N10.cz \
    -r output/mm10_with_chrL.allc.cz --show_dims 0 | head

chrom	pos	strand	context	mc	cov
chr1	3000003	+	CTG	0	0
chr1	3000005	-	CAG	0	0
chr1	3000009	+	CTA	0	0
chr1	3000016	-	CAA	0	0
chr1	3000018	-	CAC	0	0
chr1	3000019	-	CCA	0	0
chr1	3000023	+	CTT	0	0
chr1	3000027	-	CAA	0	0
chr1	3000029	-	CTC	0	0


In [16]:
! time czip query -I output/allc/FC_M_P12b_3C_2-5-M17-N10.cz \
    -r output/mm10_with_chrL.allc.cz -K chr9 -s 3000294 -e 3005294 | head

chrom	pos	strand	context	mc	cov
chr9	3000294	-	CAT	15	23
chr9	3000296	+	CCT	51	62
chr9	3000297	+	CTA	53	63
chr9	3000300	+	CAA	52	65
chr9	3000304	-	CAT	2	26
chr9	3000305	-	CCA	1	27
chr9	3000307	+	CAT	60	71
chr9	3000312	+	CTA	63	76
chr9	3000321	+	CCA	67	78

real	0m0.248s
user	0m0.129s
sys	0m0.103s


## 6. View / query a remote `.cz` (no download)

cytozip reads `.cz` files directly over HTTP Range requests when they
carry a chunk index. Below we query a remote `.cz` hosted on figshare —
only the needed chunks are fetched on-demand.


In [ ]:
! czip header -I http://gale-sapiens.salk.edu/bican/FC_M_P12b_3C_2-5-M17-N10.cz

remote demo skipped: ValueError encoding table references col 99 but only 3 columns exist


In [ ]:
! czip view -I http://gale-sapiens.salk.edu/bican/FC_M_P12b_3C_2-5-M17-N10.cz --show_dims 0 | head

In [ ]:
# query remote .cz file with local reference .cz:
! time czip query -I http://gale-sapiens.salk.edu/bican/FC_M_P12b_3C_2-5-M17-N10.cz \
    -r output/mm10_with_chrL.allc.cz -D chr9 \
    -s 3000294 -e 3005294 | head -n 5

In [ ]:
# or both the cz and the reference can be accessed via HTTP(S) URLs, without downloading any files locally:
! time czip query -I http://gale-sapiens.salk.edu/bican/FC_M_P12b_3C_2-5-M17-N10.cz -r http://gale-sapiens.salk.edu/bican/mm10_with_chrL.allc.cz -D chr9 -s 3000294 -e 3005294 | head -n 5

## 7. `catcz` and `merge_cz`

* `catcz` — concatenate per-cell `.cz` into one multi-cell `.cz` by
  adding a `cell_id` dimension.
* `merge_cz` — sum `mc` / `cov` across cells (pooled pseudobulk).


In [ ]:
! time czip catcz -O output/all_cells.cz -I "output/cz/*.cz" --add_key --title cell_id -F B,B -C mc,cov


real	0m1.432s
user	0m0.215s
sys	0m0.519s


In [18]:
! czip header -I output/all_cells.cz

magic  :  b'CZIP'
version  :  0.3
total_size  :  287336622
message  :  
formats  :  ['B', 'B']
columns  :  ['mc', 'cov']
sort_col  :  None
delta_cols  :  []
chunk_dims  :  ['chrom', 'cell_id']
header_size  :  47


In [20]:
! czip summary -I output/all_cells.cz | head

chrom	cell_id	chunk_start_offset	chunk_size	chunk_tail_offset	chunk_nblocks	chunk_nrows
chr1	FC_E17b_3C_5-5-I24-A21	47	6078973	6098344	2410	78962721
chr10	FC_E17b_3C_5-5-I24-A21	6098344	4050739	10161976	1606	52609184
chr11	FC_E17b_3C_5-5-I24-A21	10161976	3886495	14061220	1588	52027265
chr12	FC_E17b_3C_5-5-I24-A21	14061220	3666001	17739186	1490	48799752
chr13	FC_E17b_3C_5-5-I24-A21	17739186	3751420	21502555	1488	48750883
chr14	FC_E17b_3C_5-5-I24-A21	21502555	3725735	25240543	1526	49987736
chr15	FC_E17b_3C_5-5-I24-A21	25240543	3213655	28464555	1289	42230765
chr16	FC_E17b_3C_5-5-I24-A21	28464555	2977266	31451370	1188	38899643
chr17	FC_E17b_3C_5-5-I24-A21	31451370	2915036	34376011	1195	39153472


In [21]:
# merge 9 single-cell .cz files into a pseudobulk .cz file, summing the mc and cov values across all cells:
! time czip merge_cz -i output/cz -O output/pseudobulk.cz -r output/mm10_with_chrL.allc.cz -F H,H -t 20

2026-04-24 23:31:09.821 | INFO     | cytozip.merge:merge_cz:228 - output/pseudobulk.cz
2026-04-24 23:36:24.275 | INFO     | cytozip.merge:merge_cz:388 - Removing temp dir /anvil/projects/x-mcb130189/Wubin/Github/cytozip/cytozip_example_data/output/pseudobulk.cz.tmp

real	5m14.849s
user	74m3.520s
sys	8m40.996s


In [23]:
! czip view -I output/pseudobulk.cz --show_dims 0 -r output/mm10_with_chrL.allc.cz | awk '$6 >0' | head

chrom	pos	strand	context	mc	cov
chr1	3000139	-	CAA	1	1
chr1	3000142	-	CAT	1	1
chr1	3000146	-	CAG	1	1
chr1	3000161	-	CTA	1	1
chr1	3000162	-	CCT	1	1
chr1	3000163	-	CCC	1	1
chr1	3000164	-	CCC	1	1
chr1	3000169	-	CAT	1	1
chr1	3000170	-	CCA	1	1


## 8. Build a cell × gene `AnnData`

`cytozip.features.cz_to_anndata` aggregates per-cell `.cz` files over
a feature interval set. When `features=` is a GTF path, cytozip
extracts one interval per gene, merges GENCODE records sharing a
symbol, and extends each interval by `flank_bp` (default 2 kb) on
both sides.


In [18]:
from cytozip.features import cz_to_anndata

cz_paths = sorted(str(p) for p in CZ_DIR.glob('*.cz'))
print(f'{len(cz_paths)} per-cell .cz files')

t0 = time.perf_counter()
ad_gene = cz_to_anndata(
    cz_inputs=cz_paths,
    reference=str(REF_CZ),
    features=str(GTF),
    flank_bp=2000,
    score='frac',
    threads=20,
)
print(f'cz_to_anndata(GTF): {time.perf_counter()-t0:.1f}s   {ad_gene}')
print('\nobs head:'); print(ad_gene.obs.head(3))
print('\nvar head:'); print(ad_gene.var.head(3))
ad_gene.write_h5ad(OUT / 'bg_gene.h5ad')
print('wrote', OUT / 'bg_gene.h5ad')


9 per-cell .cz files


cz_to_anndata(GTF): 62.4s   AnnData object with n_obs × n_vars = 9 × 55228
    obs: 'alpha', 'beta', 'prior_mean'
    var: 'chrom', 'start', 'end', 'gene_id', 'gene_name', 'gene_type', 'strand'
    uns: 'cytozip_score'
    layers: 'mc', 'cov'

obs head:
                              alpha      beta  prior_mean
FC_E17b_3C_5-5-I24-A21    16.560707  4.300688    0.793845
FC_M_E15a_3C_1-1-I5-B1     2.186391  2.139432    0.505428
FC_M_P12b_3C_2-5-M17-N10   2.996044  2.876152    0.510208

var head:
              chrom    start      end               gene_id      gene_name  \
name                                                                         
4933401J01Rik  chr1  3071252  3076322  ENSMUSG00000102693.1  4933401J01Rik   
Gm26206        chr1  3100015  3104125  ENSMUSG00000064842.1        Gm26206   
Xkr4           chr1  3203900  3673498  ENSMUSG00000051951.5           Xkr4   

                    gene_type strand  
name                                  
4933401J01Rik             TEC     